In [30]:
# %pip install gensim

In [31]:
from gensim.models import Word2Vec

corpus = [
    'i love nlp',
    'i love machine learning!',
    'nlp is fun^^',
    'machine learning is powerful',
    'i enjoy deep learning',
    'natural langauge processing is interesting'
]
import re
sentences = [re.findall(r'\w+',text.lower()) for text in corpus]
# word2vec 모델 학습
model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=3,  # 주변단어 범위
    min_count=1 ,  #최소등장횟수
    sg = 1  # 0: CBOW  1=skip-gram
)
# 단어백터 확인
vector = model.wv['love']
# 유사단어 찾기
model.wv.most_similar('learning')


[('fun', 0.13724106550216675),
 ('machine', 0.06804423034191132),
 ('langauge', 0.0338103212416172),
 ('love', 0.009400710463523865),
 ('enjoy', 0.008315925486385822),
 ('nlp', 0.004496959038078785),
 ('powerful', -0.0036535137332975864),
 ('is', -0.010894794017076492),
 ('i', -0.023647431284189224),
 ('natural', -0.09577619284391403)]

In [32]:
# 유사도계산
model.wv.similarity('fun','learning')

0.13724104

In [33]:
model.wv.index_to_key
model.save('customword2vec.model')
loaded_model = Word2Vec.load('customword2vec.model')

In [34]:
# 한국어 지원
import pandas as pd
df = pd.read_csv('daum_movie_review.csv')
corpus = df['review'][:100]
corpus = corpus.to_numpy()

In [35]:
import re
from konlpy.tag import Okt
okt = Okt()
# 전처리
sentences = [re.sub(r'[^가-힣\s]','',text) for text in corpus]
# 형태소분리
def korean_token(text):
    return [word for word,_  in okt.pos(text,stem=True) if len(word)> 1]
    

sentences = [korean_token(doc) for doc in sentences]
# 나머지는 동일하게 적용
# word2vec 모델 학습
model = Word2Vec(
    sentences=sentences,
    vector_size=1000,
    window=3,  # 주변단어 범위
    min_count=5 ,  #최소등장횟수
    sg = 1  # 0: CBOW  1=skip-gram
)
# 유사단어 찾기
model.wv.most_similar('영화')



[('가다', 0.10001435875892639),
 ('있다', 0.08331058919429779),
 ('다음', 0.0739576667547226),
 ('같다', 0.06599210202693939),
 ('모르다', 0.055154312402009964),
 ('자다', 0.05486886575818062),
 ('나오다', 0.050617605447769165),
 ('그래도', 0.04825279489159584),
 ('재밌다', 0.04429187625646591),
 ('정말', 0.025044545531272888)]

In [36]:
# 감정분석 딥러닝

# 다음리뷰데이터 로드
# 평점 최저 최고 점수 의 절반을 threshold로 정하고 0(긍정) 1(부정)로 추가 컬럼 생성
import pandas as pd
df = pd.read_csv('daum_movie_review.csv')
df.head()

,review,rating,date,title
0,돈 들인건 티가 나지만 보는 내내 하품만,1,2018.10.29,인피니티 워
1,몰입할수밖에 없다. 어렵게 생각할 필요없다. 내가 전투에 참여한듯 손에 땀이남.,10,2018.10.26,인피니티 워
2,이전 작품에 비해 더 화려하고 스케일도 커졌지만.... 전국 맛집의 음식들을 한데 ...,8,2018.10.24,인피니티 워
3,이 정도면 볼만하다고 할 수 있음!,8,2018.10.22,인피니티 워
4,재미있다,10,2018.10.20,인피니티 워


In [37]:
df['rating'].max(), df['rating'].min()

(10, 0)

In [38]:
df['target'] =  df['rating'].apply(lambda x : 0 if x > 5 else 1)

In [39]:
# 전처리
import re
from konlpy.tag import Okt
from sklearn.feature_extraction.text import TfidfVectorizer
df['cleaned_review'] = df['review'].apply(lambda x : re.sub(r'[^가-힣\s]','',x) )
# 토큰화(한글은 형태소)
okt = Okt()
def kor_tokenize(text):
    return [word for word, _ in okt.pos(text, stem=True)]

tfidf = TfidfVectorizer(tokenizer=kor_tokenize,max_features=5000)
x_tfidf = tfidf.fit_transform(df['cleaned_review']).toarray()
y = df['target'].values

c:\Users\Playdata\AppData\Local\miniconda3\envs\new_01\lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [40]:
from torch.utils.data import DataLoader, Dataset
import torch
class TfidfDataset(Dataset):
    def __init__(self, vectors, labels):
        self.vectors = torch.FloatTensor(vectors)
        self.labels = torch.FloatTensor(labels)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, index):
        return self.vectors[index], self.labels[index]

In [41]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x_tfidf, y, random_state=42, test_size=0.2)
train_loader = DataLoader(TfidfDataset(x_train, y_train), batch_size=32, shuffle=True)
test_loader = DataLoader(TfidfDataset(x_test, y_test), batch_size=32, shuffle=False)

In [42]:
import torch.nn as nn 
class TfidfMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(hidden_dim, 1),
        )
    def forward(self, x):
        return self.network(x)

In [43]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cpu'

In [44]:
from torch.optim import Adam

model = TfidfMLP(input_dim=x_train.shape[1], hidden_dim=64).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = Adam(model.parameters(), lr=1e-3)

In [46]:
from tqdm import tqdm
epochs = 10
for epoch in tqdm(range(epochs)):
    model.train()
    local_loss = 0.0
    for vecs, labels in train_loader:
        vecs, labels = vecs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(vecs).squeeze(1), labels)
        loss.backward()
        optimizer.step()
        local_loss += loss.item()
    print(f'epoch : {epoch+1}/{epochs} loss : {(local_loss / len(train_loader)):.4f}')

 10%|█         | 1/10 [00:04<00:41,  4.64s/it]

epoch : 1/10 loss : 0.5075


 20%|██        | 2/10 [00:09<00:37,  4.69s/it]

epoch : 2/10 loss : 0.3290


 30%|███       | 3/10 [00:14<00:33,  4.81s/it]

...

epoch : 8/10 loss : 0.1575


 90%|█████████ | 9/10 [00:47<00:05,  5.61s/it]

epoch : 9/10 loss : 0.1459


100%|██████████| 10/10 [00:53<00:00,  5.36s/it]

epoch : 10/10 loss : 0.1308


In [47]:
# 예측 / 정확도 측정
model.eval()
correct = 0.0; total = 0.0
with torch.no_grad():
    for vecs, labels in test_loader:
        vecs, labels = vecs.to(device), labels.to(device)
        preds = (torch.sigmoid(model(vecs)).squeeze(1) > 0.5).float()
        correct += (preds == labels).sum().item()
        total += labels.shape[0]
    print(f'test accuracy : {correct / total:.4f}')


test accuracy : 0.8394
